# config-observ-lib — Working with observ-libs

Observ-libs are reusable Jsonnet libraries that mixins (or your own jsonnet code) `import`. They don't render to anything on their own; they're build-time dependencies. The workspace vendors them under the top-level `libs:` key.

Sections:

1. List every `libs:` entry across `/config/*.yaml`.
2. Add an observ-lib block.
3. Sync libs and inspect what was vendored.
4. Catalog of common observ-libs from `grafana/jsonnet-libs`.

In [ ]:
CONFIG_DIR = '/config'
PICK_CONFIG = 'example-observ-lib'   # one of the .yaml basenames in CONFIG_DIR

## 1. What's defined across all configs?

In [ ]:
import os, glob, yaml, pandas as pd

rows = []
for path in sorted(glob.glob(f'{CONFIG_DIR}/*.yaml')):
    cfg = yaml.safe_load(open(path)) or {}
    env = cfg.get('name', os.path.basename(path))
    for name, body in (cfg.get('libs') or {}).items():
        src = (body or {}).get('source', {})
        kind = next(iter(src.keys()), '')
        url = src.get('git', {}).get('url') or src.get('directory', {}).get('path', '')
        ref = src.get('git', {}).get('ref', '')
        include = src.get('includePaths') or []
        rows.append({'env': env, 'lib': name, 'source': kind, 'url': url, 'ref': ref,
                     'include': ','.join(include)[:60]})

df = pd.DataFrame(rows)
print(f'{len(df)} library entries across {df["env"].nunique() if not df.empty else 0} environments')
df

## 2. Add an observ-lib

Drop a block under `libs:` in your env config:

```yaml
libs:
  postgres-observ-lib:
    source:
      git:
        url: https://github.com/grafana/jsonnet-libs.git
        ref: master
        depth: 1
      includePaths:
        - postgres-observ-lib/**/*
      newRootPath: postgres-observ-lib
```

There is **no** `config:` block on a lib entry — libraries are pure jsonnet, not renderable resources. The mixin that consumes the library `import`s it from the vendored tree.

## 3. Sync libs and inspect what was vendored

`sync-mixins` handles both `mixins:` and `libs:`.

In [ ]:
import os, subprocess

env_name = PICK_CONFIG
cfg_path = f'{CONFIG_DIR}/{env_name}.yaml'
src_dir = f'/source/{env_name}'

shenv = os.environ | {'CONFIG_FILE': cfg_path, 'BUILD_DIR': src_dir}
subprocess.run(['bash', '-lc', 'sync-mixins'], env=shenv, check=False)

In [ ]:
rows = []
lib_root = f'{src_dir}/mixins'
if os.path.isdir(lib_root):
    for entry in sorted(os.listdir(lib_root)):
        full = os.path.join(lib_root, entry)
        if not os.path.isdir(full):
            continue
        if entry.endswith('-mixin'):
            continue   # mixins covered in config-mixin.ipynb
        n_libsonnet = sum(1 for _, _, files in os.walk(full) for f in files if f.endswith('.libsonnet'))
        n_json = sum(1 for _, _, files in os.walk(full) for f in files if f.endswith('.json'))
        rows.append({'lib': entry, 'libsonnet_files': n_libsonnet, 'json_files': n_json})

pd.DataFrame(rows)

## 4. Catalog of common observ-libs

Most curated `*-observ-lib` modules live in [grafana/jsonnet-libs](https://github.com/grafana/jsonnet-libs).

In [ ]:
catalog = pd.DataFrame([
    ('windows-observ-lib',                'Windows host monitoring helpers'),
    ('golang-observ-lib',                 'Go runtime metrics dashboards'),
    ('jvm-observ-lib',                    'JVM heap / GC / threads dashboards'),
    ('kafka-observ-lib',                  'Kafka broker / topic dashboards'),
    ('opentelemetry-collector-observ-lib','OTel Collector pipeline dashboards'),
    ('postgres-observ-lib',               'PostgreSQL dashboards'),
    ('process-observ-lib',                'Per-process metrics (process_exporter)'),
    ('snmp-observ-lib',                   'SNMP exporter dashboards'),
    ('alerts-observ-lib',                 'Reusable Prometheus alert templates'),
    ('common-lib',                        'Shared dashboard / panel helpers'),
    ('logs-lib',                          'Loki / log panel helpers'),
    ('mixin-utils',                       'Helpers for writing mixins'),
    ('status-panels-lib',                 'Status overview / SLO panels'),
], columns=['lib', 'description'])
catalog['source'] = 'grafana/jsonnet-libs//' + catalog['lib']
catalog[['lib', 'source', 'description']]

### Where to go next

- See [`config-mixin.ipynb`](config-mixin.ipynb) for the mixins that consume these libraries.
- See [`config-grafana-dashboard.ipynb`](config-grafana-dashboard.ipynb) to add static `.json` dashboards alongside mixin output.